In [ ]:
pip install torchmetrics # install the torchmetrics package

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 15.9 MB/s eta 0:00:00


## Generate a synthetic data

In [ ]:
from torch import nn
import torch
from torch.nn.functional import softmax

In [ ]:
def generate_synthetic_problem():
  true_weight = torch.randn(3, 10, requires_grad=False) # this is the ground truth
  training_feature_matrix = torch.randn(10, 400, requires_grad=False)
  test_feature_matrix = torch.randn(10, 400, requires_grad=False)
  training_scores = true_weight @ training_feature_matrix
  training_probabilities = softmax(
      training_scores,
      dim=0 # dimension over which softmax will be computed
  )
  test_scores = true_weight @ test_feature_matrix
  test_probabilities = softmax(
      test_scores,
      dim=0 # dimension over which softmax will be computed
  )
  # generate labels according to the probability distribution training_probabilities
  training_labels = torch.multinomial(training_probabilities.T, num_samples=1)
  training_labels = training_labels[:,0] # turn matrix into vector
  # generate labels according to the probability distribution test_probabilities
  test_labels = torch.multinomial(test_probabilities.T, num_samples=1)
  test_labels = test_labels[:,0] # turn matrix into vector
  return training_feature_matrix, training_labels, test_feature_matrix, test_labels

In [ ]:
torch.manual_seed(2)
training_feature_matrix, training_labels, test_feature_matrix, test_labels = generate_synthetic_problem()

## Fit a multinomial logistic regression model using gradient descent with no bias term

For simplicity here we assume that there is no bias in the score calculation, i.e.,
$$
f(x_i, W) = W x_i
$$
Our training objective is
$$
\min_{W} -\sum_{i=1}^n \log(p_{y_i}(W x_i))
$$
where $p$ is the softmax function.

In [ ]:
from torchmetrics.classification import MulticlassAccuracy
weight = torch.zeros(3, 10, requires_grad=True)
loss_fn = nn.CrossEntropyLoss() # sum_i -\log(p_{y_i}(   ))
accuracy_metric = MulticlassAccuracy(num_classes=3)

# use pytorch's imbuilt (stochastic) gradient descent optimizer
optimizer = torch.optim.SGD(
    [weight], # learnable parameters
    lr=0.5 # step size for gradient descent
)

print("training model with gradient descent ...")
print("#it | train objective | test objective | train accuracy | test accuracy |")

for iteration in range(11):
    # Compute prediction and loss
    training_scores = (weight @ training_feature_matrix).T
    training_objective = loss_fn(training_scores, training_labels)

    if iteration % 2 == 0: # every two iterations print out some information
      probabilities = softmax(
          training_scores,
          dim=0 # dimension over which softmax will be computed
      )
      training_accuracy = accuracy_metric(training_scores, training_labels)
      test_scores = (weight @ test_feature_matrix).T
      test_objective = loss_fn(test_scores, test_labels)
      test_accuracy = accuracy_metric(test_scores, test_labels)
      print(f"{iteration} \t {(training_objective.item()):.4f} \t {(test_objective.item()):.4f} \t {(training_accuracy):.4f} \t {(test_accuracy):.4f}") #" \t {(test_objective.item()):3f} \t {(test_accuracy):3f}")

    optimizer.zero_grad() # zero the gradient to prevent gradient accumulation
    training_objective.backward() # backpropagation
    optimizer.step() # takes a gradient descent step, i.e., x = x - lr * gradient

training model with gradient descent ...
#it | train objective | test objective | train accuracy | test accuracy |
0 	 1.0986 	 1.0986 	 0.3333 	 0.3333
2 	 0.8602 	 0.8856 	 0.7927 	 0.7540
4 	 0.7394 	 0.7789 	 0.7975 	 0.7617
6 	 0.6700 	 0.7181 	 0.8024 	 0.7640
8 	 0.6257 	 0.6796 	 0.8024 	 0.7614
10 	 0.5951 	 0.6533 	 0.8047 	 0.7664
